In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
plt.rcParams["figure.figsize"] = (12, 5)

In [ ]:
n = 1500
time = pd.date_range("2026-01-01", periods=n, freq="min")
minute = np.arange(n)
daily_cycle = np.sin(2 * np.pi * minute / 1440)
short_cycle = np.sin(2 * np.pi * minute / 120)

telemetry = pd.DataFrame({
    "timestamp": time,
    "latency_ms": 95 + 18 * daily_cycle + 8 * short_cycle + rng.normal(0, 7, n),
    "cpu_pct": 40 + 12 * daily_cycle + rng.normal(0, 5, n),
    "requests_per_min": 420 + 75 * short_cycle + rng.normal(0, 22, n),
    "error_rate": np.clip(rng.normal(0.012, 0.005, n), 0, 1),
})
events = [(205, 230), (615, 635), (1030, 1060)]
for start, end in events:
    telemetry.loc[start:end, "latency_ms"] += rng.uniform(70, 130)
    telemetry.loc[start:end, "cpu_pct"] += rng.uniform(22, 38)
    telemetry.loc[start:end, "error_rate"] += rng.uniform(0.05, 0.12)
telemetry["cpu_pct"] = telemetry["cpu_pct"].clip(0, 100)
telemetry.head()

In [ ]:
features = ["latency_ms", "cpu_pct", "requests_per_min", "error_rate"]
X = telemetry[features].copy()
scaled = RobustScaler().fit_transform(X)

model = IsolationForest(
    n_estimators=250,
    max_samples="auto",
    contamination=0.06,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
telemetry["anomaly"] = model.fit_predict(scaled) == -1
telemetry["anomaly_score"] = -model.decision_function(scaled)

median = X.median()
mad = (X - median).abs().median().replace(0, 1e-9)
robust_deviation = (X - median).abs().div(1.4826 * mad)
telemetry["primary_signal"] = robust_deviation.idxmax(axis=1)
telemetry["severity"] = pd.cut(
    telemetry["anomaly_score"],
    bins=[-np.inf, 0, 0.04, np.inf],
    labels=["normal", "warning", "critical"],
)
summary = pd.DataFrame({
    "metric": ["observations", "flagged_anomalies", "anomaly_rate_pct", "critical_events"],
    "value": [len(telemetry), int(telemetry["anomaly"].sum()),
              round(100 * telemetry["anomaly"].mean(), 2),
              int((telemetry["severity"] == "critical").sum())],
})
summary

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True, constrained_layout=True)
for ax, field, label in [
    (axes[0], "latency_ms", "Response latency (ms)"),
    (axes[1], "cpu_pct", "CPU utilization (%)"),
]:
    ax.plot(telemetry["timestamp"], telemetry[field], lw=1.0, label=label)
    flagged = telemetry[telemetry["anomaly"]]
    ax.scatter(flagged["timestamp"], flagged[field], s=13, c="crimson",
               label="Detected anomaly", zorder=3)
    ax.set_ylabel(label)
    ax.grid(alpha=0.2)
    ax.legend(loc="upper right")
axes[1].set_xlabel("Time")
plt.show()

top_events = (
    telemetry.loc[telemetry["anomaly"],
                  ["timestamp", "anomaly_score", "primary_signal"] + features]
    .sort_values("anomaly_score", ascending=False)
    .head(12)
    .reset_index(drop=True)
)
display(top_events)

In [ ]:
def detect_incident_windows(frame: pd.DataFrame, gap_minutes: int = 3) -> pd.DataFrame:
    flagged = frame.loc[frame["anomaly"], ["timestamp", "anomaly_score", "primary_signal"]].copy()
    if flagged.empty:
        return pd.DataFrame(columns=["start", "end", "observations", "peak_score", "dominant_signal"])
    gap = flagged["timestamp"].diff().dt.total_seconds().div(60)
    flagged["incident_id"] = gap.gt(gap_minutes).fillna(True).cumsum()
    incidents = (
        flagged.groupby("incident_id")
        .agg(start=("timestamp", "min"),
             end=("timestamp", "max"),
             observations=("timestamp", "size"),
             peak_score=("anomaly_score", "max"),
             dominant_signal=("primary_signal", lambda s: s.mode().iloc[0]))
        .sort_values("peak_score", ascending=False)
        .reset_index(drop=True)
    )
    return incidents

incidents = detect_incident_windows(telemetry)
display(incidents.head(15))
print(f"Processed {len(telemetry):,} synthetic observations; detected {len(incidents)} incident windows.")